<a href="https://colab.research.google.com/github/kite121/Machine-Learning-Course/blob/main/Lab12_Stacking_Cifar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 12 - Ensemble Learning.

Try to make an ensemble model for  CIFAR100 using multiple pretrained models
from [here](https://pytorch.org/vision/stable/models.html)

Use voting classifier to aggregate the predictions.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models


In [2]:
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [3]:




train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])


In [4]:
train_dataset = datasets.CIFAR100(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR100(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

100%|██████████| 169M/169M [00:13<00:00, 12.9MB/s]


In [5]:
pin_memory = torch.cuda.is_available()

train_dataloader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=pin_memory,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=pin_memory,
)


In [6]:
resnet_weights = models.ResNet18_Weights.IMAGENET1K_V1
efficientnet_weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
mobilenet_weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2

first_model = models.resnet18(weights=resnet_weights)
second_model = models.efficientnet_b0(weights=efficientnet_weights)
third_model = models.mobilenet_v3_large(weights=mobilenet_weights)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 205MB/s]


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 185MB/s]


In [7]:
first_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [8]:
for param in first_model.parameters():
    param.requires_grad = False

first_model.fc = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(first_model.fc.in_features, 100),
)

for param in first_model.fc.parameters():
    param.requires_grad = True

first_model = first_model.to(device)


In [9]:
second_model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [10]:
for param in second_model.parameters():
    param.requires_grad = False

second_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(second_model.classifier[1].in_features, 100),
)

for param in second_model.classifier.parameters():
    param.requires_grad = True

second_model = second_model.to(device)


In [11]:
third_model

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [12]:
for param in third_model.parameters():
    param.requires_grad = False

third_model.classifier[3] = nn.Linear(third_model.classifier[3].in_features, 100)

for param in third_model.classifier.parameters():
    param.requires_grad = True

third_model = third_model.to(device)


In [13]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


In [14]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        preds = logits.argmax(dim=1)

        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return total_correct / total_samples


In [15]:
class VotingClassifier(nn.Module):
    def __init__(self, models_list, weights=None):
        super().__init__()
        self.models = nn.ModuleList(models_list)

        if weights is None:
            weights = [1.0] * len(models_list)

        weights = torch.tensor(weights, dtype=torch.float32)
        weights = weights / weights.sum()
        self.register_buffer("weights", weights)

    @torch.no_grad()
    def predict_proba(self, x):
        probs = []
        for model in self.models:
            model.eval()
            logits = model(x)
            probs.append(F.softmax(logits, dim=1))

        stacked_probs = torch.stack(probs, dim=0)
        weighted_probs = stacked_probs * self.weights.view(-1, 1, 1)
        return weighted_probs.sum(dim=0)

    @torch.no_grad()
    def predict(self, x):
        return self.predict_proba(x).argmax(dim=1)

    @torch.no_grad()
    def vote(self, x):
        return self.predict(x)


In [16]:
@torch.no_grad()
def evaluate_ensemble(ensemble, loader, device):
    ensemble.eval()
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        preds = ensemble.predict(images)

        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return total_correct / total_samples


In [17]:
models_list = [first_model, second_model, third_model]
model_names = ["ResNet18", "EfficientNet-B0", "MobileNetV3-Large"]


In [18]:
criterion = nn.CrossEntropyLoss()

optimizer1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, first_model.parameters()), lr=1e-3)
optimizer2 = torch.optim.AdamW(filter(lambda p: p.requires_grad, second_model.parameters()), lr=1e-3)
optimizer3 = torch.optim.AdamW(filter(lambda p: p.requires_grad, third_model.parameters()), lr=1e-3)


In [ ]:
EPOCHS = 3
model_specs = [
    (model_names[0], first_model, optimizer1),
    (model_names[1], second_model, optimizer2),
    (model_names[2], third_model, optimizer3),
]

best_val_scores = []

for model_name, model, optimizer in model_specs:
    print(f"\nTraining {model_name}")
    best_val = 0.0

    for epoch in range(EPOCHS):
        loss, train_acc = train_one_epoch(model, train_dataloader, optimizer, criterion, device)
        val_acc = evaluate(model, test_dataloader, device)
        best_val = max(best_val, val_acc)

        print(
            f"[{model_name}] Epoch {epoch + 1}/{EPOCHS} | "
            f"loss={loss:.4f} | train_acc={train_acc:.4f} | test_acc={val_acc:.4f}"
        )

    best_val_scores.append(best_val)

weights_sum = sum(best_val_scores)
ensemble_weights = (
    [score / weights_sum for score in best_val_scores]
    if weights_sum > 0
    else [1 / len(best_val_scores)] * len(best_val_scores)
)

ensemble = VotingClassifier(models_list, weights=ensemble_weights).to(device)
ensemble_acc = evaluate_ensemble(ensemble, test_dataloader, device)

print(f"\nValidation-based ensemble weights: {ensemble_weights}")
print(f"Ensemble test_acc={ensemble_acc:.4f}")



Training ResNet18
[ResNet18] Epoch 1/3 | loss=2.6476 | train_acc=0.3588 | test_acc=0.4867
[ResNet18] Epoch 2/3 | loss=1.9482 | train_acc=0.4831 | test_acc=0.5114
[ResNet18] Epoch 3/3 | loss=1.8432 | train_acc=0.5051 | test_acc=0.5215

Training EfficientNet-B0
[EfficientNet-B0] Epoch 1/3 | loss=2.4578 | train_acc=0.4358 | test_acc=0.5190


- What is the difference between hard and soft voting?
- Can one ensemble have different models?
- How can you get the wight for each model prediction? can you make the ensemble
learn it on its own?
- What is meta-learners?